In [ ]:
# 1. SAMM: Impordi pandas (juhendi punkt 1)
# (Märkus: supabase on vajalik vaid andmebaasist laadimisel)

# 2. JA 3. SAMM: Laadi sales tabel ja prindi shape ning head()
# Kasutame CSV faili vastavalt kasutaja eelistusele

df_sales = pd.read_csv('sales.csv')
print("Kuju (shape):", df_sales.shape)  # Näitab ridade ja veergude arvu [1]
print(df_sales.head())                 # Näitab esimesi ridu [1]
print()

# 4. SAMM: Laadi customers tabel ja prindi shape ning head()
df_customers = pd.read_csv('customers.csv')
print("--- SAMM 4: CUSTOMERS TABEL ---")
print("Kuju (shape):", df_customers.shape)
print(df_customers.head())
print()

# 5. SAMM: Liida tabelid customer_id põhjal (how='left') [1]
df_merged = pd.merge(
    df_sales,
    df_customers,
    on='customer_id',
    how='left'
)

# 6. SAMM: Prindi liidatud DataFrame'i info ja kontrolli veerge
print("--- SAMM 6: LIIDETUD DATAFRAME ---")

print("Kuju (shape):", df_merged.shape)

print("\nVeergude andmetüübid (dtypes):")
print(df_merged.dtypes)

print("\nLiidetud tabeli algus (head):")
print(df_merged.head())

# KVALITEEDIKONTROLL
required_cols = ['customer_id', 'sale_date', 'total_price', 'email']

cols_exist = all(col in df_merged.columns for col in required_cols)

print(f"\nKVALITEEDIKONTROLL: Nõutud veerud {required_cols} olemas: {cols_exist}")

In [ ]:
# Roll C: RFM Analysis — Recency, Frequency, Monetary + segmendid

# Viitekuupäev on fikseeritud, et tulemused oleksid võrreldavad
today = pd.to_datetime('2025-02-28')

# Samm 2: Arvuta Recency — viimane ostu kuupäev ja päevade arv
recency_df = df_cleaned.groupby('customer_id')['sale_date'].max().reset_index()
recency_df.columns = ['customer_id', 'last_purchase']
recency_df['recency_days'] = (today - recency_df['last_purchase']).dt.days

# Samm 3: Arvuta Frequency — ostude arv
frequency_df = df_cleaned.groupby('customer_id')['sale_id'].count().reset_index()
frequency_df.columns = ['customer_id', 'frequency']

# Samm 4: Arvuta Monetary — kogukulutus
monetary_df = df_cleaned.groupby('customer_id')['total_price'].sum().reset_index()
monetary_df.columns = ['customer_id', 'monetary']

# Samm 5: Liida R, F, M ühte tabelisse pd.merge abil
rfm = pd.merge(recency_df[['customer_id', 'recency_days']], frequency_df, on='customer_id')
rfm = pd.merge(rfm, monetary_df, on='customer_id')

# Määra skoorid 1-5 (pd.qcut jagab andmed 5 võrdsesse gruppi)
# Recency puhul on väiksem number parem, seega sildid on tagurpidi [5, 4, 3, 2, 1]
rfm['R_score'] = pd.qcut(rfm['recency_days'], 5, labels=[5, 4, 3, 2, 1], duplicates='drop')
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm['M_score'] = pd.qcut(rfm['monetary'], 5, labels=[1, 2, 3, 4, 5])

# Samm 6: Arvuta summaarne RFM skoor
rfm['RFM_Score'] = rfm['R_score'].astype(int) + rfm['F_score'].astype(int) + rfm['M_score'].astype(int)

# Segmenteerimisfunktsioon skooride põhjal
def määra_segment(score):
    if score >= 13: return 'VIP Champions'
    elif score >= 10: return 'Loyal'
    elif score >= 7: return 'Potential'
    elif score >= 4: return 'At Risk'
    else: return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(määra_segment)

# VÄLJUND: RFM kokkuvõttetabel
print("--- RFM SEGMENTIDE KOKKUVÕTE ---")
segmentide_arv = rfm['Segment'].value_counts()
segmentide_protsent = rfm['Segment'].value_counts(normalize=True) * 100

kokkuvotte_df = pd.DataFrame({
    'Klientide arv': segmentide_arv,
    'Osakaal (%)': segmentide_protsent.round(1)
})

print(kokkuvotte_df)

# Kontroll: kas kõik kliendid said segmendi?
print("\nSegmendita kliente:", rfm['Segment'].isnull().sum())

# Kuvame tabeli alguse, et näha tulemusi
print("\nNäidis kliendiandmetest koos skooridega:")
rfm.head()

In [ ]:
# Loo diagramm 3 — top 10 VIP klienti kogukulutuse järgi.
# Kasuta rfm[rfm['Segment'] == 'VIP Champions'].nlargest(10, 'monetary_value')

# 1. ETTEVALMISTUS
rfm_vip = rfm.copy()

if 'customer_id' not in rfm_vip.columns:
    rfm_vip = rfm_vip.reset_index()

segment_col = 'Advanced_Segment' if 'Advanced_Segment' in rfm_vip.columns else 'Segment'
monetary_col = 'monetary_value' if 'monetary_value' in rfm_vip.columns else 'monetary'

# 2. TOP 10 VIP klienti
top_vip = (
    rfm_vip[rfm_vip[segment_col] == 'VIP Champions']
    .nlargest(10, monetary_col)
)

# 3. Lisame kliendi nimed
customer_names = (
    df_cleaned[['customer_id', 'first_name', 'last_name']]
    .drop_duplicates()
)

top_vip = top_vip.merge(
    customer_names,
    on='customer_id',
    how='left'
)

top_vip['customer_name'] = (
    top_vip['first_name'].fillna('') + ' ' + top_vip['last_name'].fillna('')
).str.strip()

top_vip['customer_name'] = top_vip['customer_name'].replace('', pd.NA)

top_vip['customer_name'] = top_vip['customer_name'].fillna(
    top_vip['customer_id'].astype(str)
)

# 4. Loome diagrammi
fig3 = px.bar(
    top_vip,
    x=monetary_col,
    y='customer_name',
    orientation='h',
    title='UrbanStyle: TOP 10 VIP-klienti kogukulutuse järgi',
    labels={
        monetary_col: 'Kogukulutus (EUR)',
        'customer_name': ''
    },
    text=monetary_col,
    color_discrete_sequence=['#009B8D']
)

# 5. Tulpade otsas täpne summa
fig3.update_traces(
    texttemplate='€%{x:,.0f}',
    textposition='outside',
    cliponaxis=False
)

# 6. Visuaalne viimistlus
fig3.update_layout(
    plot_bgcolor='white',
    font_family='Calibri',
    showlegend=False,
    bargap=0.25,

    xaxis=dict(
        showgrid=False,
        showticklabels=False,
        title=''
    ),

    yaxis=dict(
        title='',
        autorange='reversed',
        ticklabelstandoff=15
    ),

    margin=dict(t=90, l=160, r=120, b=60)
)

fig3.show()

### Kliendibaasi jaotuse visualiseerimine (Kaalutud RFM)

In [ ]:
# Diagramm: Detailne kliendibaasi jaotus (Kaalutud RFM)
import plotly.express as px

# 1. ETTEVALMISTUS: Määrame UrbanStyle'i stiilis värvid ja segmentide järjekorra
# VIP ja Loyal kasutavad bränditoone, teised loogilist gradatsiooni
urbanstyle_colors = {
    'VIP Champions': '#009B8D',      # UrbanStyle Teal
    'Loyal Customers': '#1A1A2E',    # UrbanStyle Navy
    'Regular Customers': '#32B4A9',  # Keskmine Teal
    'New Customers': '#A5DED7',      # Hele Teal
    'At Risk': '#FF9F43',            # Oranž (Hoiatus)
    'Lost': '#FF6B6B'                # Punakas (Oht)
}

# Järjekord vastavalt Sinu edasijõudnute taseme tabelile
segment_order = ['VIP Champions', 'Loyal Customers', 'Regular Customers', 
                 'New Customers', 'At Risk', 'Lost']

# 2. ANDMETE ETTEVALMISTUS
# Arvutame klientide koguarvu dünaamiliselt pealkirja jaoks
total_customers = rfm['customer_id'].nunique() if 'customer_id' in rfm.columns else len(rfm)

# Kasutame Sinu loodud 'Advanced_Segment' veergu
advanced_counts = rfm['Advanced_Segment'].value_counts().reset_index()
advanced_counts.columns = ['Segment', 'Klientide arv']

# 3. LOO VERTIKAALNE TULPDIAGRAMM
fig_advanced = px.bar(
    advanced_counts,
    x='Segment',
    y='Klientide arv',
    category_orders={'Segment': segment_order},
    color='Segment',
    color_discrete_map=urbanstyle_colors,
    title=f"UrbanStyle: Strateegiline segmenteerimine (Kaalutud RFM, kokku {total_customers} klienti)",
    text='Klientide arv' # Kuvame arvud tulpade kohal
)

# 4. KUJUNDUSE VIIMISTLUS (Tufte põhimõtted)
fig_advanced.update_traces(textposition='outside')
fig_advanced.update_layout(
    plot_bgcolor='white',
    showlegend=False,
    xaxis_title="",
    yaxis_title="Klientide arv",
    font_family="Calibri"
)

fig_advanced.show()

**Kokkuvõte & peamised järeldused**

- UrbanStyle’i kliendibaas sisaldab kokku **2540 klienti**.

- Esialgses RFM mudelis kuulus segmenti **VIP Champions**:
  - **453 klienti**
  - ehk **17,8% kogu kliendibaasist**.

- Kaalutud RFM mudelis (Monetary 2x kaal) suurenes VIP segment:
  - **1177 kliendini**,
  mis näitab, et suur osa klientidest omab ettevõtte jaoks kõrget rahalist väärtust.

- Hajuvusdiagramm näitas, et VIP-klientide kogukulutus jääb enamasti vahemikku:
  - **20 000 € – 30 000 €**,
  samal ajal kui suurem osa ülejäänud klientidest kulutab alla:
  - **5000 €**.

- TOP 10 VIP klientide analüüs näitas, et:
  - suurimate klientide individuaalne kogukulutus ulatub ligi **28 000 euroni**.

- Analüüs kinnitas, et:
  - väike osa klientidest genereerib suure osa ettevõtte kogukäibest.

- At Risk segment sisaldab:
  - **524 klienti**,
  kelle aktiivsus on langenud ja kes võivad muutuda täielikult kaotatud klientideks.

- Potential segment sisaldab:
  - **768 klienti**,
  mis annab ettevõttele suure võimaluse kasvatada uusi lojaalseid püsikliente.


# Esitluse kokkuvõte

## Mis oli meie peamine JÄRELDUS?

UrbanStyle’i käive sõltub tugevalt VIP-klientidest, kelle kogukulutus on kordades suurem kui ülejäänud klientidel ning kes genereerivad suure osa ettevõtte kogutulust.

---

## Mis OTSUS muutuks selle põhjal?

Ettevõte peaks suunama rohkem ressursse VIP-klientide hoidmisele ja At Risk klientide taasaktiveerimisele läbi personaalsemate kampaaniate ja lojaalsusprogrammide.

---

## Mis meid ÜLLATAS?

Kõige suurem üllatus oli see, kui palju muutis tulemusi Monetary skoori kahekordne kaal ning kui suureks kasvas VIP segment kaalutud RFM mudelis.

## Soovitused Markole

### VIP programm
Rakendada VIP-klientidele:
- varajane ligipääs uutele toodetele,
- personaalsed VIP sooduskoodid,
- tasuta tarne ja lojaalsuspreemiad.

Eesmärk on suurendada lojaalsust ja säilitada kõige väärtuslikumad kliendid.

### Win-back kampaania
Käivitada At Risk klientidele:
- personaliseeritud „Me igatseme teid” e-mailid,
- piiratud ajaga sooduspakkumised,
- soovitused varasemate ostude põhjal.

Eesmärk on taastada klientide aktiivsus enne nende kaotamist.

### Nurture programm
Suunata Potential ja New Customers kliendid lojaalseteks püsiklientideks:
- onboarding e-mailide seeria,
- lojaalsuspunktid,
- motiveerivad sõnumid nagu „Üks ost VIP-staatuseni”.

Eesmärk on kasvatada korduvoste ja kliendi pikaajalist väärtust.

## Meeskonna refleksioon

### Mis oli suurim üllatus?
Kõige suurem üllatus oli see, kui suure osa ettevõtte käibest moodustavad VIP-kliendid. Samuti oli üllatav, kui palju muutis tulemusi Monetary skoori kahekordne kaal kaalutud RFM mudelis.

### Milline on meie peamine soovitus Markole?
Kõige olulisem tegevus on VIP-klientide hoidmine ning At Risk klientide kiire taasaktiveerimine. Need kaks segmenti mõjutavad ettevõtte käivet kõige rohkem.

### Milliseid andmeid oleks veel vaja?
Analüüsi oleks aidanud täiendada:
- kliendi vanus ja demograafilised andmed,
- toodete kategooriad,
- tagastuste info,
- veebikäitumine,
- lojaalsusprogrammi kasutus,
- NPS või kliendirahulolu andmed.

# Esitluse kokkuvõte Markole

Meie meeskond ehitas täieliku Python pandas töövoo UrbanStyle kliendiandmete analüüsimiseks: laadisime ja puhastasime andmed, arvutasime RFM mõõdikud ning lõime kaalutud kliendisegmenteerimise mudeli. Analüüsi käigus segmenteerisime kokku **2540 klienti** kuude erinevasse kliendisegmenti.

Peamine leid on see, et **VIP Champions** segment sisaldab ettevõtte kõige väärtuslikumaid kliente. Sellesse segmenti kuulub **516 klienti ehk 20.3% kogu kliendibaasist**, kuid nende kogukulutus on oluliselt kõrgem kui teistel segmentidel. TOP 10 VIP kliendi individuaalne kogukulutus jääb vahemikku umbes **20 000–28 000 eurot**, mis näitab väga tugevat kliendiväärtuse kontsentratsiooni.

Scatter-diagramm näitab selgelt, et kõrge väärtusega kliendid paiknevad madala recency ja kõrge monetary piirkonnas — ehk nad ostavad sageli, hiljuti ja suurtes summades. Näiteks kõige kõrgema väärtusega VIP kliendi kogukulutus ületas **27 000 eurot**.

Samas tuvastasime suure hulga **At Risk** kliente — kokku **377 klienti ehk 14.8% kliendibaasist**. Need kliendid on varem ostnud, kuid pole pikemat aega aktiivsed olnud. See on kõige kriitilisem segment churn-riskiga klientide vaatest ning nende tagasivõitmine võib anda kiire ärilise mõju.

Kaalutud RFM mudel (Monetary 2x kaal) võimaldas meil täpsemalt eristada:
- **516 VIP Champions klienti (20.3%)**
- **485 Loyal Customers klienti (19.1%)**
- **537 Regular Customers klienti (21.1%)**
- **508 New Customers klienti (20.0%)**
- **377 At Risk klienti (14.8%)**
- **117 Lost klienti (4.6%)**

---

# Peamised järeldused

## 1. Kõige väärtuslikum segment

VIP Champions kliendid genereerivad ebaproportsionaalselt suure osa käibest. TOP 10 VIP kliendi kogukulutus ületab kõigil juhtudel **20 000 euro piiri**, mis näitab väga tugevat kliendiväärtuse kontsentratsiooni.

## 2. Kiiret tähelepanu vajav segment

At Risk segment sisaldab ligi **15% kogu kliendibaasist**. Scatter-diagramm näitab, et osa neist klientidest olid varasemalt suure väärtusega ostjad, kuid nende aktiivsus on langenud.

## 3. Segmentidel peab olema erinev kommunikatsioon

Analüüs näitab, et kliendid jagunevad väga erineva käitumisega gruppidesse:
- ligi **40% klientidest** kuulub kas VIP või Loyal segmenti,
- umbes **20% klientidest** on uued kliendid,
- ligi **15% klientidest** vajab win-back tegevusi.

---

# Soovitused Markole

## VIP programm

Luua VIP Champions klientidele:
- early access kampaaniad,
- personaalsed sooduskoodid,
- lojaalsusprogramm,
- eksklusiivsed pakkumised.

Eesmärk on hoida aktiivsena segmenti, mis moodustab umbes viiendiku kliendibaasist, kuid genereerib ebaproportsionaalselt suure osa käibest.

---

## Win-back kampaania

Käivitada At Risk klientidele:
- “Me igatseme sind” e-maili seeria,
- ajaliselt piiratud sooduspakkumised,
- personaalsed soovitused varasema ostukäitumise põhjal.

377 kliendi tagasivõitmine võib anda kiire mõju käibele.

---

## New Customers onboarding

Luua New Customers segmendile:
- welcome e-maili seeria,
- esmaostu soodustused,
- cross-sell soovitused.

Eesmärk on muuta **508 uut klienti** korduvostjateks ja kasvatada lojaalsust juba varases etapis.

---

# Demo jaoks 3 lühikest lauset

## Meie peamine järeldus

516 VIP Champions klienti moodustavad UrbanStyle’i kõige väärtuslikuma segmendi ning nende hoidmine peaks olema ettevõtte prioriteet.

## Milline otsus muutuks selle põhjal?

Turundus peaks kasutama segmentidele erinevaid kampaaniaid, sest ligi 15% klientidest on churn-riskis ja vajavad win-back tegevusi.

## Mis meid üllatas?

Kõrge väärtusega At Risk kliente oli rohkem, kui algselt eeldasime — kokku 377 klienti ehk 14.8% kliendibaasist.

In [ ]:
fig.write_html('RFM.html')